<a href="https://colab.research.google.com/github/sevinchoyazimboevna-source/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Menu detector")

Menu detector


In [18]:

# ------------------
# Import Libraries
# ------------------
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
DATASET_PATH= "/content/drive/MyDrive/food101_dataset"
CUSTOM_CLASS_MAPPING = {
    "hamburger": "hamburger",
    "hot_dog": "hot_dog",
    "chocolate_cake": "dessert",  # label grouping | class consolidation
    "cheesecake": "dessert",      # label grouping | class consolidation
    "kabob": "kabob",
    "manti": "manti",
    "osh": "osh"
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kabob', 'manti', 'osh']

CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

# print(NUM_CLASSES)
# print(CLASS_TO_IDX)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 300x400, 512x512 image sizes
# 0 ~ 255
# 0.0 ~ 1.0
# 0.234
# H, W, C => C, H, W | RGB = Red, Green, Blue
# Normalize
# pixel = (pixel - mean) / std

In [ ]:
# --------------------
# Custom Dataset Class
# --------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        # print('image_path', img_path)
        label = self.labels[idx]
        # print('label', label)
        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))
        if self.transform:
            image = self.transform(image)
        return image, label

In [9]:
# --------------------
# Gather and Split Data
# --------------------
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)
    print('class_path:', class_path)
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kabob
class_path: /content/drive/MyDrive/food101_dataset/manti
class_path: /content/drive/MyDrive/food101_dataset/osh
images_length 1056
1056


In [10]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2) #thread | parallel loading for speed
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)


images_length 1056
images_length 1056


In [12]:
# pretrained model
model = mobilenet_v2(weights="IMAGENET1K_V1") # pretrained model | ligthweight | CNN | 1000 class | million
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning | backbone | model layer freeze

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 130MB/s]


In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)

device cuda


In [14]:

criterion = nn.CrossEntropyLoss() # Loss Function | '70%' burger, '30%' pilaf
optimizer = optim.Adam(model.parameters(), lr=0.001) # weight
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick | 10%-20%
# print('criterion:', criterion)
# print('optimizer:', optimizer)

In [20]:


# --------------------
# Training Loop
# --------------------

import os
import torch
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

NUM_EPOCHS = 10
best_accuracy = 0.0

save_path = '/content/menu_detector.pth' if os.path.exists('/content') else 'menu_detector.pth'

for epoch in range(NUM_EPOCHS):
    model.train()  # train mode
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        # Forward and Backward (Backpropagation)
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()  # zero the gradient
        outputs = model(images)  # Forward Pass
        loss = criterion(outputs, labels)  # Calculate Loss

        loss.backward()
        optimizer.step()  # Adam optimizer
        running_loss += loss.item()  # Track Loss

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total if total > 0 else 0.0

    avg_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0.0

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {avg_loss:.4f}, Val Accuracy: {val_acc:.2f}%")

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), save_path)
        print("Saved new best model!")


images_length 1056
images_length 1056
images_length 1056


Epoch 1/10:   0%|          | 0/33 [00:00<?, ?it/s]

images_length 1056
images_length 1056


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f6ec40b5e40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    Exception ignored in: self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7f6ec40b5e40>

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
Traceback (most recent call last):
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
if w.is_alive():    
self._shutdown_workers()  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'

AssertionError:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
can only test a child process
    if w.is_alive():
  File "/usr/lib/

images_length 1056
images_length 264
images_length 1056
images_length 1056
Epoch [1/10] Loss: 2.0127, Val Accuracy: 0.00%
images_length 1056
images_length 1056
images_length 1056


...

images_length 1056
images_length 1056
images_length 1056
images_length 264
images_length 1056
images_length 1056
Epoch [9/10] Loss: 2.0103, Val Accuracy: 0.00%
images_length 1056
images_length 1056
images_length 1056


Epoch 10/10:   0%|          | 0/33 [00:00<?, ?it/s]

images_length 1056
images_length 1056


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f6ec40b5e40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f6ec40b5e40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

images_length 1056
images_length 264
images_length 1056
images_length 1056
Epoch [10/10] Loss: 2.0145, Val Accuracy: 0.00%


In [ ]:

# --------------------
# Required Imports
# --------------------
import os
import io
import torch
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2
from PIL import Image
from google.colab import files
import matplotlib.pyplot as plt

# --------------------
# Define Semantic Classes
# --------------------
CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kabob', 'manti', 'osh']
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}

# --------------------
# Transform for Uploaded Images
# --------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --------------------
# Device
# --------------------
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print('Device:', device)

# --------------------
# Find Saved Model
# --------------------
model_path = '/content/menu_detector.pth'

if not os.path.isfile(model_path):
    model_path = 'menu_detector.pth'

if not os.path.isfile(model_path):
    print("Searching for saved model...")

    for root, dirs, filenames in os.walk('/content'):
        if 'menu_detector.pth' in filenames:
            model_path = os.path.join(
                root, 'menu_detector.pth'
            )
            break

if not os.path.isfile(model_path):
    raise FileNotFoundError(
        "menu_detector.pth not found! "
        "Run the Training Loop first and save the trained model."
    )

print("Model found:", model_path)

# --------------------
# Load Model
# --------------------
model = mobilenet_v2(weights=None)

model.classifier[1] = torch.nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

state_dict = torch.load(
    model_path,
    map_location='cpu',
    weights_only=True
)

model.load_state_dict(state_dict)

model = model.to(device)
model.eval()

print("Model loaded successfully!")

# --------------------
# Upload & Predict
# --------------------
print("Upload one or more images of your food:")

uploaded = files.upload()

for image_name, image_bytes in uploaded.items():

    try:
        image = Image.open(
            io.BytesIO(image_bytes)
        ).convert('RGB')

        # Display image
        plt.figure(figsize=(4, 4))
        plt.imshow(image)
        plt.axis('off')
        plt.title(f'Uploaded: {image_name}')
        plt.show()

        # Prepare image
        image_tensor = transform(image)
        image_tensor = image_tensor.unsqueeze(0)
        image_tensor = image_tensor.to(device)

        # Predict
        with torch.inference_mode():
            output = model(image_tensor)
            probs = torch.softmax(output, dim=1)[0]

            topk = torch.topk(
                probs,
                k=min(4, NUM_CLASSES)
            )

        # Results
        print("\nPrediction:")

        for i in range(topk.indices.size(0)):

            class_index = topk.indices[i].item()
            label = CLASSES[class_index]

            confidence = topk.values[i].item() * 100

            print(
                f"  ✅ {label}: {confidence:.2f}%"
            )

        # Best prediction
        best_index = topk.indices[0].item()
        best_label = CLASSES[best_index]
        best_confidence = topk.values[0].item() * 100

        print("\n-------------------------")
        print(f"Detected Food: {best_label}")
        print(f"Confidence: {best_confidence:.2f}%")
        print("-------------------------\n")

    except Exception as e:
        print(f"Error processing {image_name}: {e}")

print("Prediction completed!")